In [1]:
import os
import shutil
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Paths
feature_csv    = "xception_patch_mask_features_gbm.csv"
label_csv      = "GBM_labels.csv"
patch_img_root = "GBM_0067_0108"
split_root     = "svm_splits_gbm"
model_save_dir = "svm_gbm"

os.makedirs(model_save_dir, exist_ok=True)
EPOCHS = 5

In [3]:
# ==========================================
# 📊 CELL 2: DATA PREPARATION (CLEANED)
# ==========================================
# 1. Load Features
feat_df = pd.read_csv(feature_csv).drop_duplicates(subset="filename", keep="first")

# 2. Load Labels
label_df = pd.read_csv(label_csv)

# 3. Keep only the necessary columns
label_df = label_df[["filename", "label"]]

# 4. Remove rows where 'label' might contain errors or non-numeric strings
# This ensures we only have 0 or 1
label_df['label'] = pd.to_numeric(label_df['label'], errors='coerce')
label_df = label_df.dropna(subset=['label'])

# 5. Inner Merge (Keeps only filenames present in both sets)
df = pd.merge(label_df, feat_df, on="filename", how="left")

# 6. Final Clean: Remove any row with NaN features or labels
df = df.dropna()
df["label"] = df["label"].astype(int)

# 7. Extract arrays
X = df.drop(columns=["filename", "label"]).values
y = df["label"].values
filenames = df["filename"].values

print(f"✅ Data Cleaned. {len(df)} samples remaining.")
print(f"Class Distribution:\n{df['label'].value_counts()}")

/tmp/ipykernel_553978/3745353561.py:8: DtypeWarning: Columns (0: original_slide) have mixed types. Specify dtype option on import or set low_memory=False.
  label_df = pd.read_csv(label_csv)


✅ Data Cleaned. 317082 samples remaining.
Class Distribution:
label
0    315042
1      2040
Name: count, dtype: int64


In [ ]:
best_val_acc = 0.0
history = []

for epoch in range(1, 2):
    # Randomly shuffle and split (80% Train, 20% Test/Val)
    X_train, X_test, y_train, y_test, fn_train, fn_test = train_test_split(
        X, y, filenames, test_size=0.2, random_state=None, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    
    svm = SVC(kernel="rbf", C=1.0, class_weight="balanced", probability=True)
    svm.fit(X_train_scaled, y_train)
    
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"Epoch {epoch}/{EPOCHS} | Val Accuracy: {acc:.4f}")
    history.append({"epoch": epoch, "accuracy": acc})
    
    if acc > best_val_acc:
        best_val_acc = acc
        # Save the actual model files
        joblib.dump(svm, os.path.join(model_save_dir, "svm_best_model_gbm.pkl"))
        joblib.dump(scaler, os.path.join(model_save_dir, "scaler_best_gbm.pkl"))
        
        # Capture the 'best' metadata for the next cells
        best_data = {
            'fn_train': fn_train, 'fn_test': fn_test,
            'y_train': y_train, 'y_test': y_test,
            'y_pred': y_pred, 'X_test_scaled': X_test_scaled
        }

print(f"\n💾 Best model (Acc: {best_val_acc:.4f}) saved to {model_save_dir}")

In [ ]:
def organize_files(file_list, labels, split_name):
    for fname, lbl in zip(file_list, labels):
        src = os.path.join(patch_img_root, fname)
        dst = os.path.join(split_root, split_name, str(int(lbl)), fname)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(src):
            shutil.copy(src, dst)

print("📂 Organizing files into Train/Test folders based on best epoch...")
organize_files(best_data['fn_train'], best_data['y_train'], "train")
organize_files(best_data['fn_test'],  best_data['y_test'],  "test")
print("✅ Folders ready.")

In [ ]:
# Print Report
print("\n📊 BEST MODEL CLASSIFICATION REPORT:")
print(classification_report(best_data['y_test'], best_data['y_pred']))

# Confusion Matrix

print("\n🧮 CONFUSION MATRIX:")
print(confusion_matrix(best_data['y_test'], best_data['y_pred']))

# Save Patch-Level CSV
y_prob = svm.predict_proba(best_data['X_test_scaled'])[:, 1]
results_df = pd.DataFrame({
    "filename": best_data['fn_test'],
    "true_label": best_data['y_test'],
    "pred_label": best_data['y_pred'],
    "confidence": y_prob,
    "correct": (best_data['y_test'] == best_data['y_pred'])
})
results_df.to_csv("best_epoch_patch_metrics.csv", index=False)
pd.DataFrame(history).to_csv("training_history.csv", index=False)

print("\n✅ Metrics stored in best_epoch_patch_metrics.csv")